In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, RadioButtons, HBox, VBox, Layout, Output
from IPython.display import display, Markdown, HTML

display(HTML("""
<style>
.horizontal-radio .widget-radio-box {
    display: flex !important;
    flex-direction: row !important;
    flex-wrap: nowrap !important;
    gap: 25px !important;
}
.horizontal-radio .widget-radio-box label {
    margin: 0 !important;
    white-space: nowrap !important;
    font-weight: bold;
}
.horizontal-radio > label {
    display: none !important;
}
</style>
"""))

omega = np.linspace(0, np.pi, 1000)

fig, axes = plt.subplots(3, 1, figsize=(9, 6.5), sharex=True)

mag_line, = axes[0].plot(omega / np.pi, np.zeros_like(omega), color='blue', lw=2)
phase_line, = axes[1].plot(omega / np.pi, np.zeros_like(omega), color='green', lw=2)
gd_line, = axes[2].plot(omega / np.pi, np.zeros_like(omega), color='red', lw=2)

axes[0].set_ylabel('Amplitude (dB)', fontsize=10)
axes[0].grid(True, linestyle='--', alpha=0.7)

axes[1].set_ylabel('Phase (rad)', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.7)

axes[2].set_ylabel(r'Group Delay $\tau(\omega)$', fontsize=10)
axes[2].set_xlabel(r'Normalized Frequency ($\omega / \pi$)', fontsize=10)
axes[2].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()

plot_output = Output()

def update_plot(selection, r_val, theta_val):
    th = np.deg2rad(theta_val)

    total_mag_dB = np.zeros_like(omega)
    total_phase = np.zeros_like(omega)
    total_gd = np.zeros_like(omega)

    if selection == 'Single Zero':
        total_mag_dB += 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        total_phase += np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        total_gd += (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))

    elif selection == 'Single Pole':
        total_mag_dB -= 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        total_phase -= np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        total_gd -= (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))

    elif selection == 'Complex Conjugate Zeros':
        term1_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_mag_dB += term1_mag + term2_mag

        term1_ph = np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        term2_ph = np.arctan2(r_val * np.sin(omega + th), 1 - r_val * np.cos(omega + th))
        total_phase += term1_ph + term2_ph

        term1_gd = (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_gd = (r_val**2 - r_val * np.cos(omega + th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_gd += term1_gd + term2_gd

    elif selection == 'Complex Conjugate Poles':
        term1_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_mag = 10 * np.log10(1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_mag_dB -= term1_mag + term2_mag

        term1_ph = np.arctan2(r_val * np.sin(omega - th), 1 - r_val * np.cos(omega - th))
        term2_ph = np.arctan2(r_val * np.sin(omega + th), 1 - r_val * np.cos(omega + th))
        total_phase -= term1_ph + term2_ph

        term1_gd = (r_val**2 - r_val * np.cos(omega - th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega - th))
        term2_gd = (r_val**2 - r_val * np.cos(omega + th)) / (1 + r_val**2 - 2 * r_val * np.cos(omega + th))
        total_gd -= term1_gd + term2_gd

    mag_line.set_ydata(total_mag_dB)
    phase_line.set_ydata(total_phase)
    gd_line.set_ydata(total_gd)

    axes[0].set_title(f'LTI Frequency Response for: {selection}', fontsize=11, fontweight='bold')

    axes[0].relim()
    axes[0].autoscale_view(scalex=False, scaley=True)
    axes[1].relim()
    axes[1].autoscale_view(scalex=False, scaley=True)
    axes[2].relim()
    axes[2].autoscale_view(scalex=False, scaley=True)

    fig.canvas.draw_idle()

    with plot_output:
        plot_output.clear_output(wait=True)
        display(fig)

selection_widget = RadioButtons(
    options=['Single Zero', 'Single Pole', 'Complex Conjugate Zeros', 'Complex Conjugate Poles'],
    value='Complex Conjugate Zeros',
    description='',
    disabled=False,
    layout=Layout(width='100%')
)

selection_widget.add_class('horizontal-radio')

r_widget = FloatSlider(
    min=0.0,
    max=0.99,
    step=0.05,
    value=0.8,
    description='Radius (r):',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=Layout(width='20%')
)

theta_widget = FloatSlider(
    min=0.0,
    max=180.0,
    step=5.0,
    value=45.0,
    description='Angle (deg):',
    continuous_update=True,
    style={'description_width': 'initial'},
    layout=Layout(width='20%')
)

controls = VBox([
    HBox([selection_widget], layout=Layout(width='100%')),
    HBox([r_widget, theta_widget], layout=Layout(width='100%', gap='20px'))
])

def update(*args):
    update_plot(selection_widget.value, r_widget.value, theta_widget.value)

selection_widget.observe(update, names='value')
r_widget.observe(update, names='value')
theta_widget.observe(update, names='value')

display(controls)

with plot_output:
    display(fig)

display(plot_output)

update()